In [ ]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter


path_godas = r'C:\Users\jiuwe\study\data\Godas\\'       #1980-2024
path_soda = r'C:\Users\jiuwe\study\data\soda\\'         #1900-1980
path_ncep = r'C:\Users\jiuwe\study\data\necp\monthly\\' #1980-2024
path_noaa = r'C:\Users\jiuwe\study\data\noaa-20c\\'     #1900-1980
path_oa = r'C:\Users\jiuwe\study\data\OAFlux_v3\\'

lat = np.arange(-90, 91, 1)
lon = np.arange(0, 361, 1)

In [ ]:
df = pd.read_csv(r'C:\Users\jiuwe\study\data\RGTracks-20C-v1.0.2\jeremychleung-RGTracks-20C-e7c7c2e\data\v1.0\1850-2014_OWZ_RGTracks-20C_v1.csv', index_col='track_id', usecols=['track_id', 'year', 'month', 'lon', 'lat'])

In [ ]:
lat = np.arange(-90, 91, 1)
lon = np.arange(0, 361, 1)
time = pd.date_range('1851-01', '2015-01', freq='MS')
nt, ny, nx= len(time), len(lat), len(lon)
ds_tc = xr.Dataset(
    {
        'tc_grid':(['time', 'lat', 'lon'], np.zeros((nt, ny, nx)))
    },
    coords={
        'time':(['time'], time),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
for i in range(6843):
    tc = df.loc[i][['year', 'month', 'lon', 'lat']].to_numpy()
    tc_time = tc[:, :2].astype('int')
    tc_loc = tc[:, 2:]
    tc_loc[:, 0] = np.where(tc_loc[:, 0] < 0, tc_loc[:, 0]+360, tc_loc[:, 0])
    TIME = np.array([f'{tc_time[j, 0]:d}-{tc_time[j, 1]:02d}-01' for j in range(tc.shape[0])], dtype='datetime64[ns]')
    print(f'{i} {TIME[0]} to {TIME[-1]}')
    # for j in range(tc.shape[0]):
    for j in range(1):
        LON = tc_loc[j, 0]
        LAT = tc_loc[j, 1]
        IS = int(LON - 10)
        IE = int(LON + 10)
        JS = int(LAT - 5) + 90
        JE = int(LAT + 5) + 90
        if IS < 0:
            ds_tc.tc_grid.loc[TIME[j]][JS:JE, IS:-1] += 1
            IS = 0
        if IE > 360:
            ds_tc.tc_grid.loc[TIME[j]][JS:JE, :IE-360] += 1
            IE = -1
        ds_tc.tc_grid.loc[TIME[j]][JS:JE, IS:IE] += 1
# ds_tc.to_netcdf('./data/NOAA-20C.tc_grid.nc')
ds_tc.to_netcdf('./data/NOAA-20C.tcg_grid.nc')

In [ ]:
(month+12) % 12 + 1

In [ ]:
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp < np.datetime64('2014-01')]
ep = ep[ep < np.datetime64('2014-01')]
mix = mix[mix < np.datetime64('2014-01')]
suc = suc[suc < np.datetime64('2014-01')]
events = [mix, ep, cp, suc]
tcg_grid = xr.open_dataset('./data-20c/NOAA-20C_TC_grid.nc').tcg_grid.rolling(lat=5, lon=5, center=True, min_periods=1).mean()
tct_grid = xr.open_dataset('./data-20c/NOAA-20C_TCT_grid.nc').tct_grid.rolling(lat=5, lon=5, center=True, min_periods=1).mean()
ssta = xr.open_dataset('./data/ssta.nc').sst

rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()

# tcg_grid = tcg_grid.where(tcg_grid != 0)
# tct_grid = tct_grid.where(tct_grid != 0)
maskg = np.all(tcg_grid == 0, axis=0)
maskt = np.all(tct_grid == 0, axis=0)
tcg_grid -= tcg_grid.groupby('time.month').map(rolling)
tct_grid -= tct_grid.groupby('time.month').map(rolling)
ssta -= ssta.groupby('time.month').map(rolling)

month = np.arange(-6, 30)
nt, ny, nx = len(month), len(lat), len(lon)

ds_nino = xr.Dataset(
    {
        'tcg':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'tct':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'sst':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'tcg_p':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'tct_p':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'sst_p':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
    },
    coords={
        'type':(['type'], ['MIX', 'EP', 'CP', 'SUC']),
        'month':(['month'], month),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)

sst_group = ssta.groupby('time.month')
tcg_group = tcg_grid.groupby('time.month')
tct_group = tct_grid.groupby('time.month')
month_t = (month+12) % 12 + 1
# for i in range(4):
#     for j in range(len(month)):
#         event_tcgs = tcg_grid.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')]
#         event_tcts = tct_grid.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')]
#         event_ssts = ssta.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')]

#         ds_nino.tcg[i, j] = event_tcgs.mean(dim='time')
#         ds_nino.tct[i, j] = event_tcts.mean(dim='time')
#         ds_nino.sst[i, j] = event_ssts.mean(dim='time')
        
#         sst_mon = sst_group[month_t[j]]
#         tcg_mon = tcg_group[month_t[j]]
#         tct_mon = tct_group[month_t[j]]
#         _, ds_nino.tcg_p[i, j] = stats.ttest_ind(sst_mon, event_ssts, equal_var=False, nan_policy='propagate')
#         _, ds_nino.tct_p[i, j] = stats.ttest_ind(tcg_mon.where(tcg_mon != 0), event_tcgs.where(tcg_mon != 0), equal_var=False, nan_policy='propagate')
#         _, ds_nino.sst_p[i, j] = stats.ttest_ind(tct_mon.where(tcg_mon != 0), event_ssts.where(tcg_mon != 0), equal_var=False, nan_policy='propagate')
        
# ds_nino.to_netcdf('./data-20c/composite_tc_sst.nc')


In [ ]:
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp >= np.datetime64('1960-01')]
ep = ep[ep >= np.datetime64('1960-01')]
mix = mix[mix >= np.datetime64('1960-01')]
suc = suc[suc >= np.datetime64('1960-01')]
events = [mix, ep, cp, suc]

ds_tc = xr.open_dataset(r'C:\Users\jiuwe\study\data\TC\2025\TC_grid.nc')
tcg_grid = ds_tc.tcg_grid
tct_grid = ds_tc.tct_grid
ssta = xr.open_dataset('./data/sst.nc').sst

rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()

tcg_grid -= tcg_grid.groupby('time.month').map(rolling)
tct_grid -= tct_grid.groupby('time.month').map(rolling)
ssta -= ssta.groupby('time.month').map(rolling)

month = np.arange(-6, 30)
nt, ny, nx = len(month), len(lat), len(lon)

ds_nino = xr.Dataset(
    {
        'tcg':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'tct':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'sst':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'tcg_p':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'tct_p':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'sst_p':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
    },
    coords={
        'type':(['type'], ['MIX', 'EP', 'CP', 'SUC']),
        'month':(['month'], month),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)

            
sst_cli = ssta.mean(dim='time')
tcg_cli = tcg_grid.mean(dim='time')
tct_cli = tct_grid.mean(dim='time')
for i in range(4):
    for j in range(len(month)):
        event_tcgs = tcg_grid.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')]
        event_tcts = tct_grid.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')]
        event_ssts = ssta.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')]
        
        ds_nino.tcg[i, j] = event_tcgs.mean(dim='time')
        ds_nino.tct[i, j] = event_tcts.mean(dim='time')
        ds_nino.sst[i, j] = event_ssts.mean(dim='time')
        
        # ds_nino.tcg_p[i, j] = ttest(event_tcgs, tcg_cli)
        # ds_nino.tct_p[i, j] = ttest(event_tcts, tct_cli)
        # ds_nino.sst_p[i, j] = ttest(event_ssts, sst_cli)
        
ds_nino.to_netcdf('./data-20c/composite_tc_sst_obs.nc')


In [ ]:
# composite wind
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp < np.datetime64('2009-01')]
ep = ep[ep < np.datetime64('2009-01')]
mix = mix[mix < np.datetime64('2009-01')]
suc = suc[suc < np.datetime64('2009-01')]
events = [mix, ep, cp, suc]


In [ ]:
ds_u = xr.open_dataset(path_noaa+'uwnd.mon.mean.tp.nc')
ds_v = xr.open_dataset(path_noaa+'vwnd.mon.mean.tp.nc')

# level = np.array([1000, 950, 900, 850, 800, 700, 500, 300, 100, 50])
level = 850
uwnd = ds_u.uwnd.loc[:, level, :, :]
vwnd = ds_v.vwnd.loc[:, level, :, :]
rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()

uwnd -= uwnd.groupby('time.month').map(rolling)
vwnd -= vwnd.groupby('time.month').map(rolling)

month = np.arange(-6, 30)
lat, lon = uwnd.lat.values, uwnd.lon.values
nt, ny, nx = len(month), len(lat), len(lon)

ds_wind = xr.Dataset(
    {
        'uwnd':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
        'vwnd':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
    },
    coords={
        'type':(['type'], ['MIX', 'EP', 'CP', 'SUC']),
        'month':(['month'], month),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)

for i in range(4):
    for j in range(len(month)):
        event_uwnd = uwnd.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')]
        event_vwnd = vwnd.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')]

        ds_wind.uwnd[i, j] = event_uwnd.mean(dim='time')
        ds_wind.vwnd[i, j] = event_vwnd.mean(dim='time')

ds_wind.to_netcdf('./data-20c/composite_wind.nc')

In [ ]:
events[i] + np.timedelta64(month[j], "M")

In [ ]:
def interp(tc):
    tc_grid = np.zeros((len(lat), len(lon)))
    for i in range(tc.shape[0]):
        LON = tc[i, 0]
        LAT = tc[i, 1]
        IS = int(LON - 10)
        IE = int(LON + 10)
        JS = int(LAT - 5) + 90
        JE = int(LAT + 5) + 90
        if IS < 0:
            tc_grid[JS:JE, IS:-1] += 1
            IS = 0
        if IE > 360:
            tc_grid[JS:JE, :IE-360] += 1
            IE = -1
        tc_grid[JS:JE, IS:IE] += 1
    return tc_grid
    
lat = np.arange(-90, 91, 1)
lon = np.arange(0, 361, 1)
tc = df.loc[0:8][['lon', 'lat']].to_numpy()
tc[:, 0] = np.where(tc[:, 0] < 0, tc[:, 0]+360, tc[:, 0])
tc_itp = ds_tc.tc_grid.mean(dim='time').values

levels = np.linspace(0, 10, 21)
colors = ['white','cyan', 'lightgreen', 'yellow', 'orange', 'red']
cmap = LinearSegmentedColormap.from_list('my_cmap', colors, N=len(levels)-1)
norm = BoundaryNorm(levels, len(levels)-1)


fig, ax = plt.subplots(1, 1, figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
# fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.set_extent([0, 360, -90, 90], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#F5F5F5', lw=0.1, zorder=1)
ax.coastlines(lw=0.8, color='k')
ax.set_yticks(np.arange(-90, 91, 30), crs=ccrs.PlateCarree())
ax.set_xticks(np.arange(0, 361, 40), crs=ccrs.PlateCarree())   
ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
ax.yaxis.set_major_formatter(LatitudeFormatter())
cb = ax.contourf(lon, lat, tc_itp, cmap=cmap, levels=levels, transform=ccrs.PlateCarree())
fig.colorbar(cb)

In [ ]:
ds_tc.tc_grid.loc[np.array([f'{int(tc[i, 0]):d}-{int(tc[i, 1]):02d}-01' for i in range(10)], dtype='datetime64[ns]')] += 1

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
# fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.set_extent([0, 360, -90, 90], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#F5F5F5', lw=0.1, zorder=1)
ax.coastlines(lw=0.8, color='k')
ax.set_yticks(np.arange(-90, 91, 30), crs=ccrs.PlateCarree())
ax.set_xticks(np.arange(0, 361, 40), crs=ccrs.PlateCarree())   
ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
ax.yaxis.set_major_formatter(LatitudeFormatter())
for tc in range(0, 6000):
    lons = df.loc[tc]['lon'].to_numpy()[::2]
    lats = df.loc[tc]['lat'].to_numpy()[::2]
    dif = np.diff(lons)
    jump = np.where(abs(dif) > 180)[0]
    if len(jump) == 0:
        lon_360 = lons
        ax.plot(lon_360, lats, color='cyan', linewidth=0.6, transform=ccrs.PlateCarree(central_longitude=0), zorder=1)
        ax.scatter(lon_360[0], lats[0], color='orange', s=2, transform=ccrs.PlateCarree(central_longitude=0), zorder=2)
    else:
        lon_360 = np.where(lons <= 0, lons + 360, lons)
        ax.plot(lon_360, lats, color='cyan', linewidth=0.6, transform=ccrs.PlateCarree(central_longitude=180), zorder=1)
        ax.scatter(lon_360[0], lats[0], color='orange', s=2, transform=ccrs.PlateCarree(central_longitude=180), zorder=2)
    
    # print(lons)
    # ax.plot(lons, lats, color='r', linewidth=0.8)



In [ ]:
pt = xr.open_dataset(path_soda+'pottmp.mon.mean.nc').pottmp.loc[:, :50]
obs_pt = xr.open_dataset(path_godas+'pottmp.mon.mean.nc').pottmp.loc[:, :50] - 273.15
ML_pt = xr.concat([pt[:, :].mean(dim='level'), obs_pt.mean(dim='level')], dim='time')
ML_T = ML_pt.mean(dim='time', skipna=False)
ML_t = ML_pt - ML_T

In [ ]:
xr.open_dataset(path_soda+'soda_temp.nc')

In [ ]:
# combine Godas
lon = np.arange(120, 281)
lat = np.arange(-40, 41)
level_t = np.arange(10, 210, 10)
level_w = np.arange(5, 200, 10)
time_obs = pd.date_range('1980-01-01', '2025-01-01', freq='M')
nt = 45*12
nz = len(level_t)
ny = 81
nx = 161
ds_pottmp = xr.Dataset(
    {
        'pottmp':(['time', 'level', 'lat', 'lon'], np.empty((nt, nz, ny, nx)))
    },
    coords={
        'time':(['time'], time_obs),
        'level':(['level'], level_t),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
ds_u = xr.Dataset(
    {
        'ucur':(['time', 'level', 'lat', 'lon'], np.empty((nt, nz, ny, nx)))
    },
    coords={
        'time':(['time'], time_obs),
        'level':(['level'], level_t),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
ds_v = xr.Dataset(
    {
        'vcur':(['time', 'level', 'lat', 'lon'], np.empty((nt, nz, ny, nx)))
    },
    coords={
        'time':(['time'], time_obs),
        'level':(['level'], level_t),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
ds_w = xr.Dataset(
    {
        'w':(['time', 'level', 'lat', 'lon'], np.empty((nt, nz, ny, nx)))
    },
    coords={
        'time':(['time'], time_obs),
        'level':(['level'], level_w),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
for i in range(45):
    pottmp = xr.open_dataset(path_godas+rf'pottmp\pottmp.{i+1980}.nc').pottmp[:, :, ::3, :].loc[:, :200, -41:41, 119:281].interp(lat=lat, lon=lon)
    ucur = xr.open_dataset(path_godas+rf'u_current\ucur.{i+1980}.nc').ucur[:, :, ::3, :].loc[:, :200, -41:40, 120:280]
    vcur = xr.open_dataset(path_godas+rf'v_current\vcur.{i+1980}.nc').vcur[:, :, ::3, :].loc[:, :200, -41:40, 120:280]
    dzdt = xr.open_dataset(path_godas+rf'dzdt\dzdt.{i+1980}.nc').dzdt[:, :, ::3, :].loc[:, :200, -41:41, 119:281].interp(lat=lat, lon=lon)
    
    ds_pottmp.pottmp[12*i:12*(i+1), :, :, :] = pottmp.values
    ds_u.ucur[12*i:12*(i+1), :, :, :] = ucur.values
    ds_v.vcur[12*i:12*(i+1), :, :, :] = vcur.values
    ds_w.w[12*i:12*(i+1), :, :, :] = dzdt.values
    



In [ ]:
ds_pottmp.to_netcdf(path_godas+'pottmp.mon.mean.nc')
ds_u.to_netcdf(path_godas+'ucur.mon.mean.nc')
ds_v.to_netcdf(path_godas+'vcur.mon.mean.nc')
ds_w.to_netcdf(path_godas+'dzdt.mon.mean.nc')

In [ ]:
# combine soda
lon = np.arange(120, 281)
lat = np.arange(-40, 41)
level = [5.01,  15.07,  25.28, 35.76, 46.61, 57.98]
time_20c = pd.date_range('1900-01-01', '1980-01-01', freq='M')
nt = len(time_20c)
nz = len(level)
ny = 81
nx = 161

ds_pottmp = xr.Dataset(
    {
        'pottmp':(['time', 'level', 'lat', 'lon'], np.empty((nt, nz, ny, nx)))
    },
    coords={
        'time':(['time'], time_20c),
        'level':(['level'], level),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
ds_u = xr.Dataset(
    {
        'ucur':(['time', 'level', 'lat', 'lon'], np.empty((nt, nz, ny, nx)))
    },
    coords={
        'time':(['time'], time_20c),
        'level':(['level'], level),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
ds_v = xr.Dataset(
    {
        'vcur':(['time', 'level', 'lat', 'lon'], np.empty((nt, nz, ny, nx)))
    },
    coords={
        'time':(['time'], time_20c),
        'level':(['level'], level),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
ds_w = xr.Dataset(
    {
        'w':(['time', 'level', 'lat', 'lon'], np.empty((nt, nz, ny, nx)))
    },
    coords={
        'time':(['time'], time_20c),
        'level':(['level'], level),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)

pottmp = xr.open_dataset(path_soda+'soda_temp.nc').temp[:-12].loc[:, 0:60, -41:41, 119:281].interp(latitude=lat, longitude=lon)
ucur = xr.open_dataset(path_soda+'soda_ucur.nc').u[:-12].loc[:, 0:60, -41:41, 119:281].interp(latitude=lat, longitude=lon)
vcur = xr.open_dataset(path_soda+'soda_vcur.nc').v[:-12].loc[:, 0:60, -41:41, 119:281].interp(latitude=lat, longitude=lon)
w = xr.open_dataset(path_soda+'soda_w.nc').w[:-12].loc[:, 0:60, -41:41, 119:281].interp(latitude=lat, longitude=lon)

ds_pottmp.pottmp[:, :, :, :] = pottmp.values
ds_u.ucur[:, :, :, :] = ucur.values
ds_v.vcur[:, :, :, :] = vcur.values
ds_w.w[:, :, :, :] = w.values

ds_pottmp.to_netcdf(path_soda+'pottmp.mon.mean.nc')
ds_u.to_netcdf(path_soda+'ucur.mon.mean.nc')
ds_v.to_netcdf(path_soda+'vcur.mon.mean.nc')
ds_w.to_netcdf(path_soda+'dzdt.mon.mean.nc')


In [ ]:
# combine heat flux
lon = np.arange(120, 281)
lat = np.arange(-40, 41)
time_all = pd.date_range('1900-01', '2025-01', freq='M')
nt = len(time_all)
ny = len(lat)
nx = len(lon)

ds_net_Q = xr.Dataset(
   {
       'sw':(['time', 'lat', 'lon'], np.empty((nt, ny, nx))),
       'lw':(['time', 'lat', 'lon'], np.empty((nt, ny, nx))),
       'sh':(['time', 'lat', 'lon'], np.empty((nt, ny, nx))),
       'lh':(['time', 'lat', 'lon'], np.empty((nt, ny, nx))),
   },
   coords={
        'time':(['time'], time_all),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)

obs_ds = xr.open_dataset(path_ncep+'dswrf.sfc.mon.mean.nc').dswrf.loc['1980':'2024', 41:-41, 120:282][:, ::-1, :]
obs_dl = xr.open_dataset(path_ncep+'dlwrf.sfc.mon.mean.nc').dlwrf.loc['1980':'2024', 41:-41, 120:282][:, ::-1, :]
obs_us = xr.open_dataset(path_ncep+'uswrf.sfc.mon.mean.nc').uswrf.loc['1980':'2024', 41:-41, 120:282][:, ::-1, :]
obs_ul = xr.open_dataset(path_ncep+'ulwrf.sfc.mon.mean.nc').ulwrf.loc['1980':'2024', 41:-41, 120:282][:, ::-1, :]

obs_sh = xr.open_dataset(path_ncep+'shtfl.sfc.mon.mean.nc').shtfl.loc['1980':'2024', 41:-41, 120:282][:, ::-1, :]
obs_lh = xr.open_dataset(path_ncep+'lhtfl.sfc.mon.mean.nc').lhtfl.loc['1980':'2024', 41:-41, 120:282][:, ::-1, :]

obs_net_s = obs_ds - obs_us
obs_net_l = obs_dl - obs_ul

ds = xr.open_dataset(path_noaa+'dswrf.sfc.mon.mean.tp.nc').dswrf[:-12]
dl = xr.open_dataset(path_noaa+'dlwrf.sfc.mon.mean.tp.nc').dlwrf[:-12]
us = xr.open_dataset(path_noaa+'uswrf.sfc.mon.mean.tp.nc').uswrf[:-12]
ul = xr.open_dataset(path_noaa+'ulwrf.sfc.mon.mean.tp.nc').ulwrf[:-12]
sh = xr.open_dataset(path_noaa+'shtfl.mon.mean.tp.nc').shtfl[:-12]
lh = xr.open_dataset(path_noaa+'lhtfl.mon.mean.tp.nc').lhtfl[:-12]

net_s = ds - us
net_l = dl - ul


ds_net_Q.sw[:80*12, :, :] = net_s.values
ds_net_Q.lw[:80*12, :, :] = net_l.values
ds_net_Q.sh[:80*12, :, :] = -sh.values
ds_net_Q.lh[:80*12, :, :] = -lh.values

ds_net_Q.sw[80*12:, :, :] = obs_net_s.interp(lat=lat, lon=lon).values
ds_net_Q.lw[80*12:, :, :] = obs_net_l.interp(lat=lat, lon=lon).values
ds_net_Q.sh[80*12:, :, :] = -obs_sh.interp(lat=lat, lon=lon).values
ds_net_Q.lh[80*12:, :, :] = -obs_lh.interp(lat=lat, lon=lon).values

ds_net_Q.to_netcdf('./data/Qnet.nc')


In [ ]:
# mixed layer heat budget equation
# take the mixed layer depth equal to 45m
# unit: K/month
lon = np.arange(120, 281)
lat = np.arange(-40, 41)
time_all = pd.date_range('1900-01', '2025-01', freq='M')
nt = len(time_all)
ny = len(lat)
nx = len(lon)

ds_dTdt = xr.Dataset(
   {
       'MLHB':(['time', 'term', 'lat', 'lon'], np.empty((nt, 15, ny, nx))),
   },
   coords={
        'time':(['time'], time_all),
        'term':(['term'], ['dT/dt', '-Utx', '-uTx', '-utx', '-Vty', '-vTy', '-vty', '-Wtz', '-wTz', '-wtz', 'SW', 'LW', 'SH', 'LH', 'R']),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
a = 6.4e+6
# compute dT/dt
skipt = 80*12
pt = xr.open_dataset(path_soda+'pottmp.mon.mean.nc').pottmp.loc[:, :50].mean(dim='level')
obs_pt = xr.open_dataset(path_godas+'pottmp.mon.mean.nc').pottmp.loc[:, :50].mean(dim='level') - 273.15
ML_pt = xr.concat([pt, obs_pt], dim='time')
ML_T = ML_pt.copy()
ML_T[:skipt] = ML_pt[:skipt].mean(dim='time')
ML_T[skipt:] = ML_pt[skipt:].mean(dim='time')
# ML_T = ML_pt.mean(dim='time', skipna=False)
ML_t = ML_pt - ML_T
ds_dTdt.MLHB[:, 0, :, :] = ML_t.assign_coords(time=np.arange(1500)).differentiate('time').values

# compute zonal adv
fac = 30*24*3600/(np.pi/180*a)
ucur = xr.open_dataset(path_soda+'ucur.mon.mean.nc').ucur.loc[:, :50].mean(dim='level')
obs_ucur = xr.open_dataset(path_godas+'ucur.mon.mean.nc').ucur.loc[:, :50].mean(dim='level')
U = ucur.mean(dim='time')
obs_U = obs_ucur.mean(dim='time')
u = ucur - U
obs_u = obs_ucur - obs_U
ds_dTdt.MLHB[:skipt, 1, :, :] = -U.values * ML_t[:skipt].differentiate('lon').values*fac
ds_dTdt.MLHB[:skipt, 2, :, :] = -u.values * ML_T[:skipt].differentiate('lon').values*fac
ds_dTdt.MLHB[:skipt, 3, :, :] = -u.values * ML_t[:skipt].differentiate('lon').values*fac
ds_dTdt.MLHB[skipt:, 1, :, :] = -obs_U.values * ML_t[skipt:].differentiate('lon').values*fac
ds_dTdt.MLHB[skipt:, 2, :, :] = -obs_u.values * ML_T[skipt:].differentiate('lon').values*fac
ds_dTdt.MLHB[skipt:, 3, :, :] = -obs_u.values * ML_t[skipt:].differentiate('lon').values*fac

# compute meridional adv
vcur = xr.open_dataset(path_soda+'vcur.mon.mean.nc').vcur.loc[:, :50].mean(dim='level')
obs_vcur = xr.open_dataset(path_godas+'vcur.mon.mean.nc').vcur.loc[:, :50].mean(dim='level')
V = vcur.mean(dim='time')
obs_V = obs_vcur.mean(dim='time')
v = vcur - V
obs_v = obs_vcur - obs_V
ds_dTdt.MLHB[:skipt, 4, :, :] = -V.values * ML_t[:skipt].differentiate('lat').values*fac
ds_dTdt.MLHB[:skipt, 5, :, :] = -v.values * ML_T[:skipt].differentiate('lat').values*fac
ds_dTdt.MLHB[:skipt, 6, :, :] = -v.values * ML_t[:skipt].differentiate('lat').values*fac
ds_dTdt.MLHB[skipt:, 4, :, :] = -obs_V.values * ML_t[skipt:].differentiate('lat').values*fac
ds_dTdt.MLHB[skipt:, 5, :, :] = -obs_v.values * ML_T[skipt:].differentiate('lat').values*fac
ds_dTdt.MLHB[skipt:, 6, :, :] = -obs_v.values * ML_t[skipt:].differentiate('lat').values*fac

# compute vertical adv
fac = 31*24*3600
pt = xr.open_dataset(path_soda+'pottmp.mon.mean.nc').pottmp.loc[:, :60]
obs_pt = xr.open_dataset(path_godas+'pottmp.mon.mean.nc').pottmp.loc[:, :55] - 273.15
T = pt[:, :].mean(dim='time')
t = pt - T
obs_T = obs_pt[:, :].mean(dim='time')
obs_t = obs_pt - obs_T
dzdt = xr.open_dataset(path_soda+'dzdt.mon.mean.nc').w.loc[:, :50].mean(dim='level')
obs_dzdt = xr.open_dataset(path_godas+'dzdt.mon.mean.nc').w.loc[:, :50].mean(dim='level')
W = dzdt.mean(dim='time', skipna=False)
obs_W = obs_dzdt.mean(dim='time', skipna=False)
w = dzdt - W
obs_w = obs_dzdt - obs_W
ds_dTdt.MLHB[:80*12, 7, :, :] = -(t[:, :-1].mean(dim='level') - t[:, -1])/(-t.level[:-1].mean()+t.level[-1]) * W * fac
ds_dTdt.MLHB[80*12:, 7, :, :] = -(obs_t[:, :-1].mean(dim='level') - obs_t[:, -1])/(-obs_t.level[:-1].mean()+obs_t.level[-1]) * obs_W * fac
ds_dTdt.MLHB[:80*12, 8, :, :] = -w*(T[:-1].mean(dim='level') - T[-1])/(T.level[-1]-T.level[:-1].mean()) * fac
ds_dTdt.MLHB[80*12:, 8, :, :] = -obs_w*(obs_T[:-1].mean(dim='level') - obs_T[-1])/(obs_T.level[-1]-obs_T.level[:-1].mean()) * fac
ds_dTdt.MLHB[:80*12, 9, :, :] = -w*(t[:, :-1].mean(dim='level') - t[:, -1])/(-t.level[:-1].mean()+t.level[-1]) * fac
ds_dTdt.MLHB[80*12:, 9, :, :] = -obs_w*(obs_t[:, :-1].mean(dim='level') - obs_t[:, -1])/(-obs_t.level[:-1].mean()+obs_t.level[-1]) * fac

# compute Qnet
ds_Qnet = xr.open_dataset('./data/Qnet.nc')
fac = 45*1025*4096/(30*24*3600)
ds_dTdt.MLHB[:skipt, 10, :, :] = (ds_Qnet.sw[:skipt] - ds_Qnet.sw[:skipt].mean(dim='time')).values/fac
ds_dTdt.MLHB[:skipt, 11, :, :] = (ds_Qnet.lw[:skipt] - ds_Qnet.lw[:skipt].mean(dim='time')).values/fac
ds_dTdt.MLHB[:skipt, 12, :, :] = (ds_Qnet.sh[:skipt] - ds_Qnet.sh[:skipt].mean(dim='time')).values/fac
ds_dTdt.MLHB[:skipt, 13, :, :] = (ds_Qnet.lh[:skipt] - ds_Qnet.lh[:skipt].mean(dim='time')).values/fac
ds_dTdt.MLHB[skipt:, 10, :, :] = (ds_Qnet.sw[skipt:] - ds_Qnet.sw[skipt:].mean(dim='time')).values/fac
ds_dTdt.MLHB[skipt:, 11, :, :] = (ds_Qnet.lw[skipt:] - ds_Qnet.lw[skipt:].mean(dim='time')).values/fac
ds_dTdt.MLHB[skipt:, 12, :, :] = (ds_Qnet.sh[skipt:] - ds_Qnet.sh[skipt:].mean(dim='time')).values/fac
ds_dTdt.MLHB[skipt:, 13, :, :] = (ds_Qnet.lh[skipt:] - ds_Qnet.lh[skipt:].mean(dim='time')).values/fac

ds_dTdt.to_netcdf('./data/dTdt.nc')

In [ ]:
fig, axs = plt.subplots(3 ,1, figsize=(14, 12))
mlhb = ds_dTdt.MLHB.loc[:, :, -5:5, 180:280].mean(dim=['lat', 'lon'])
datas = [mlhb.loc[:'1940'], mlhb.loc['1940':'1980'], mlhb.loc['1980':]]
for i in range(3):
    ax = axs[i]
    data = datas[i]
    ax.plot(data.time, data[:, 0], color='k', label='dTdt')
    # ax.plot(mlhb.time, mlhb[:, 1:4].sum(dim='term'), label='adv u')
    # ax.plot(mlhb.time, mlhb[:, 4:7].sum(dim='term'), label='adv v')
    # ax.plot(mlhb.time, mlhb[:, 7:10].sum(dim='term'), label='adv w')
    # ax.plot(data.time, data[:, 1:14].sum(dim='term'), color='r', label='mlhb')
    ax.plot(data.time, data[:, 1:10].sum(dim='term'), label='adv')
    # ax.plot(data.time, data[:, 10:14].sum(dim='term'), label='hflx')
    ax.legend(loc='upper left')
# fig.savefig('./pic/time series mlhb')

In [ ]:
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
# cp = cp[cp >= np.datetime64('1955-01')]
# ep = ep[ep >= np.datetime64('1955-01')]
# mix = mix[mix >= np.datetime64('1955-01')]
# suc = suc[suc >= np.datetime64('1955-01')]
events = [mix, ep, cp, suc]

In [ ]:
mlhb = ds_dTdt.MLHB.loc[:, :, -5:5, 180:230].mean(dim=['lat', 'lon']).assign_coords(time=pd.date_range('1900-01-01', '2024-12-01', freq='MS'))
fig, axs = plt.subplots(4, 1, figsize=(12, 12))
for i in range(4):
    ax = axs[i]
    data = np.zeros((15))
    for j in range(len(events[i])):
        data[:] += mlhb.loc[events[i][j]:events[i][j]+np.timedelta64(7, 'M')].mean(dim='time').values/len(events[i])
    data[-1] = data[0] - data[1:-1].sum()
    ax.bar(mlhb.term, data, width=0.4, color='#4A90D9', edgecolor='white', linewidth=0.5,)
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-', zorder=5)
    ax.set_ylim([-0.2, 0.3])
fig.savefig('./pic/bar.png', dpi=300)

In [ ]:
np.timedelta64